# Part 7 — Running noise backwards: the true denoising step

_Rigorous Courses · Diffusion Models — Part 7 of 12_

**Bayes' rule plus complete-the-square turns the noising recipe around — the exact one-step denoiser is a Gaussian you can write down**

You will see with your own eyes why the $x_0$-free reverse step is out of reach (a two-bump histogram no Gaussian can imitate), then put the lesson's exact posterior formulas for $\tilde\mu_t$ and $\tilde\beta_t$ on trial against 200,000 brute-force simulated chains, verify the hand example to four decimals, and finish with the oracle sampler: pure noise reassembling into an 8-cluster ring, one exact Gaussian draw at a time.

---

This notebook accompanies the lesson. Run cells top to bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## The step we wish we could take

Generation needs the reverse conditional $q(x_{t-1} \mid x_t)$: given the current noisy point, where was the chain one step earlier? Bayes' rule offers a route,

$$ q(x_{t-1} \mid x_t) = \frac{q(x_t \mid x_{t-1})\, q(x_{t-1})}{q(x_t)}, $$

but the marginals $q(x_{t-1})$ and $q(x_t)$ are averages over the data distribution $q(x_0)$ — the very thing the whole course is trying to learn. The lesson's claim: for a two-value dataset ($x_0 = \pm 2$, equal odds) the honest reverse conditional is not even Gaussian — it has **two bumps**, one per possible origin. Let's watch that happen.

### Step 1 — Build the two schedules

We need the teaching schedule of the running example — $T = 3$ with $\beta = (0.1,\ 0.2,\ 0.3)$, so $\alpha = (0.9,\ 0.8,\ 0.7)$ and $\bar\alpha = (0.9,\ 0.72,\ 0.504)$ — and the real-size $T = 200$ linear schedule from part 6. The `_full` arrays store $\bar\alpha_t$ at index $t$, including the convention $\bar\alpha_0 = 1$ (before any step, all of the signal survives). The asserts pin the teaching values and the always-falling shape.

In [ ]:
betas3 = np.array([0.1, 0.2, 0.3])
alphas3 = 1.0 - betas3
abar3 = np.cumprod(alphas3)
abar3_full = np.concatenate(([1.0], abar3))

T = 200
betas = np.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
abar = np.cumprod(alphas)
abar_full = np.concatenate(([1.0], abar))

print("teaching schedule abar:", abar3)
print(f"real schedule: abar_1 = {abar[0]:.6f}   abar_T = {abar[-1]:.6f}")

assert np.allclose(abar3, [0.9, 0.72, 0.504])
assert np.all(np.diff(abar) < 0)
assert np.all((abar > 0) & (abar < 1))

### Step 2 — Simulate 200,000 forward chains from a two-point dataset

Half the clean points sit at $x_0 = -2$, half at $x_0 = +2$. Every chain runs the full teaching schedule step by step, and we keep every intermediate value — this pile of (start, path) records is the raw material for every check below. The variance assert reuses part 6's bookkeeping: for this data ($\mathrm{Var}(x_0) = 4$), the step-$t$ variance must be $\bar\alpha_t \cdot 4 + (1-\bar\alpha_t)$.

In [ ]:
n = 200_000
x0 = rng.choice(np.array([-2.0, 2.0]), size=n)

eps1 = rng.standard_normal(n)
x1 = np.sqrt(alphas3[0]) * x0 + np.sqrt(betas3[0]) * eps1
eps2 = rng.standard_normal(n)
x2 = np.sqrt(alphas3[1]) * x1 + np.sqrt(betas3[1]) * eps2
eps3 = rng.standard_normal(n)
x3 = np.sqrt(alphas3[2]) * x2 + np.sqrt(betas3[2]) * eps3

var_pred = abar3[2] * 4.0 + (1.0 - abar3[2])

print(f"fraction starting at +2: {(x0 > 0).mean():.4f}")
print(f"Var(x_3) empirical = {x3.var():.4f}   predicted = {var_pred:.4f}")

assert set(np.unique(x0)) == {-2.0, 2.0}
assert abs((x0 > 0).mean() - 0.5) < 0.01
assert abs(x3.var() - var_pred) < 0.05

### Step 3 — The two bumps: q(x_1 | x_2) without knowing the start

Condition on exactly what a generator would see: $x_2$ near the lesson's value $0.1$ (a window $|x_2 - 0.1| \le 0.4$). The histogram of where those chains sat at step 1 has two clearly separated bumps — one for chains that drifted down from $+2$, one for chains that drifted up from $-2$ — and their sizes track how plausible each origin is. No single Gaussian can imitate this shape, and computing the bump sizes needs the data distribution. This is the object we cannot have.

In [ ]:
window = np.abs(x2 - 0.1) <= 0.4
x1_win = x1[window]

frac_up = (x1_win > 0.5).mean()
frac_down = (x1_win < -0.5).mean()
frac_valley = (np.abs(x1_win) <= 0.5).mean()

plt.figure(figsize=(7, 4))
plt.hist(x1_win, bins=40, density=True, color="#4ea1ff")
plt.xlabel("x_1")
plt.ylabel("density")
plt.title("q(x_1 | x_2 near 0.1): two bumps, one per possible origin")
plt.show()

print(f"chains in the window: {window.sum()}")
print(f"upper bump mass = {frac_up:.3f}   lower bump mass = {frac_down:.3f}   valley mass = {frac_valley:.3f}")

assert window.sum() > 500
assert frac_up > 0.10 and frac_down > 0.10
assert frac_valley < 0.05

## The fix: condition on the clean point too

The lesson's move: also condition on the start. Inside the context "given $x_0$", Bayes' rule plus the Markov drop $q(x_t \mid x_{t-1}, x_0) = q(x_t \mid x_{t-1})$ leaves a product of part 6's Gaussians, and part 3's complete-the-square recipe collapses it to the destination formulas of the whole part:

$$ q(x_{t-1} \mid x_t, x_0) = \mathcal{N}\big(x_{t-1};\ \tilde\mu_t(x_t, x_0),\ \tilde\beta_t\big), \qquad \tilde\beta_t = \frac{(1-\bar\alpha_{t-1})\,\beta_t}{1-\bar\alpha_t}, $$

$$ \tilde\mu_t = a_t\, x_0 + b_t\, x_t, \qquad a_t = \frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}, \qquad b_t = \frac{\sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}. $$

Exactly Gaussian — no approximation, for any step size. Everything below puts these three formulas on trial.

### Step 4 — Implement the formulas and rebuild the lesson's weight table

One small function returns the triple $(a_t,\ b_t,\ \tilde\beta_t)$ for any schedule. We use it to rebuild the lesson's table on the teaching schedule and check all nine numbers to four decimals — including the $t = 1$ row, where the convention $\bar\alpha_0 = 1$ forces $a_1 = 1$, $b_1 = 0$, $\tilde\beta_1 = 0$: a zero-spread spike at exactly $x_0$.

In [ ]:
def posterior_coeffs(t, betas_arr, alphas_arr, abar_full_arr):
    """Return (a_t, b_t, beta_tilde_t) for a schedule, with 1-indexed step t."""
    noise_before = 1.0 - abar_full_arr[t - 1]
    noise_after = 1.0 - abar_full_arr[t]
    a_t = np.sqrt(abar_full_arr[t - 1]) * betas_arr[t - 1] / noise_after
    b_t = np.sqrt(alphas_arr[t - 1]) * noise_before / noise_after
    beta_tilde = noise_before * betas_arr[t - 1] / noise_after
    return a_t, b_t, beta_tilde

table = np.array([posterior_coeffs(t, betas3, alphas3, abar3_full) for t in [1, 2, 3]])

print("t    a_t (weight on x0)    b_t (weight on x_t)    beta_tilde_t")
for t, row in zip([1, 2, 3], table):
    print(f"{t}    {row[0]:.4f}                {row[1]:.4f}                 {row[2]:.4f}")

lesson_table = np.array([[1.0, 0.0, 0.0], [0.6776, 0.3194, 0.0714], [0.5132, 0.4723, 0.1694]])

assert np.allclose(table, lesson_table, atol=5e-5)

### Step 5 — Brute force vs formula: 200,000 chains can't be wrong

Now the main event. Fix the origin ($x_0 = +2$) and select the simulated chains whose $x_2$ landed within $\pm 0.05$ of the hand example's observation $1.1$. For each selected chain the formula predicts its own center $\tilde\mu_2 = a_2 \cdot 2 + b_2\, x_2$, so the **residuals** $x_1 - \tilde\mu_2$ must be centered at 0 with variance $\tilde\beta_2$ — and the selected $x_1$ histogram must trace the predicted bell. That is the whole theorem, tested against nature.

In [ ]:
a2, b2, bt2 = posterior_coeffs(2, betas3, alphas3, abar3_full)

sel = (x0 == 2.0) & (np.abs(x2 - 1.1) <= 0.05)
x1_sel = x1[sel]
mu_sel = a2 * 2.0 + b2 * x2[sel]
resid = x1_sel - mu_sel

print(f"selected chains: {sel.sum()}")
print(f"residual mean = {resid.mean():+.4f}   (predicted 0)")
print(f"residual var  = {resid.var():.4f}   (predicted beta_tilde_2 = {bt2:.4f})")

assert sel.sum() > 1000
assert abs(resid.mean()) < 0.02
assert abs(resid.var() - bt2) < 0.01

grid = np.linspace(x1_sel.min(), x1_sel.max(), 200)
mu_center = a2 * 2.0 + b2 * 1.1
pdf = np.exp(-(grid - mu_center) ** 2 / (2 * bt2)) / np.sqrt(2 * np.pi * bt2)

plt.figure(figsize=(7, 4))
plt.hist(x1_sel, bins=50, density=True, alpha=0.6, label="simulated x_1 (x_0 = +2, x_2 near 1.1)")
plt.plot(grid, pdf, "k--", label="predicted N(mu_tilde, beta_tilde)")
plt.xlabel("x_1")
plt.ylabel("density")
plt.title("the true posterior: formula vs 200,000-chain brute force")
plt.legend()
plt.show()

## A posterior you can compute on paper

The lesson computed, for $x_0 = 2$ and observed $x_2 = 1.1$: $\tilde\mu_2 = 1.7066$ and $\tilde\beta_2 = 0.0714$. The code must reproduce all four decimals. We then run the lesson's consistency check: feed the formula the **average** $x_2$ given $x_0$, namely $\sqrt{\bar\alpha_2}\, x_0$, and it must hand back the average $x_1$, namely $\sqrt{\bar\alpha_1}\, x_0$ — the two odd-looking weights conspire to keep averages exactly consistent.

### Step 6 — The hand example, to four decimals

In [ ]:
mu2_hand = a2 * 2.0 + b2 * 1.1

print(f"a_2 = {a2:.6f}   b_2 = {b2:.6f}")
print(f"mu_tilde_2   = {mu2_hand:.4f}   (lesson: 1.7066)")
print(f"beta_tilde_2 = {bt2:.4f}   (lesson: 0.0714)")

assert abs(mu2_hand - 1.7066) < 5e-5
assert abs(bt2 - 0.0714) < 5e-5

x2_avg = np.sqrt(abar3_full[2]) * 2.0
mu_at_avg = a2 * 2.0 + b2 * x2_avg
x1_avg = np.sqrt(abar3_full[1]) * 2.0

print(f"feed the average x_2 = {x2_avg:.6f}: the formula returns {mu_at_avg:.6f}")
print(f"the average x_1 given x_0 = 2 is                         {x1_avg:.6f}")

assert np.isclose(mu_at_avg, x1_avg)

## The tug-of-war between two anchors

On the teaching schedule the handover from the $x_t$ anchor to the $x_0$ anchor happens in three big jerks. On a production-size schedule it is a long, quiet drama: $b_t$ hugs 1 for almost the whole chain, and only the last few steps swing hard toward $x_0$.

### Step 7 — Plot both weights across the T = 200 schedule

Left: $a_t$ and $b_t$ over all 200 steps. Read it right to left — generation order, $t = 200 \to 1$ — and you see the posterior trusting the current position for most of the walk, with the clean anchor taking over completely at the very end. Right: $\tilde\beta_t$ sits strictly below $\beta_t$ at every step — knowing the start shrinks every step's uncertainty, exactly as the ratio $(1-\bar\alpha_{t-1})/(1-\bar\alpha_t) < 1$ demands.

In [ ]:
t_axis = np.arange(1, T + 1)
coeffs = np.array([posterior_coeffs(t, betas, alphas, abar_full) for t in t_axis])
a_w = coeffs[:, 0]
b_w = coeffs[:, 1]
bt_w = coeffs[:, 2]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(t_axis, a_w, label="a_t (weight on x0)")
ax1.plot(t_axis, b_w, label="b_t (weight on x_t)")
ax1.set_xlabel("step t")
ax1.set_ylabel("weight")
ax1.set_title("the two anchors across the schedule (T = 200)")
ax1.legend()

ax2.plot(t_axis, betas, label="beta_t (forward dose)")
ax2.plot(t_axis, bt_w, label="beta_tilde_t (posterior spread)")
ax2.set_xlabel("step t")
ax2.set_ylabel("variance")
ax2.set_title("knowing x0 shrinks every step")
ax2.legend()

plt.tight_layout()
plt.show()

print(f"t = 1:   a = {a_w[0]:.4f}   b = {b_w[0]:.4f}   beta_tilde = {bt_w[0]:.6f}")
print(f"t = 100: a = {a_w[99]:.4f}   b = {b_w[99]:.4f}   beta_tilde = {bt_w[99]:.6f}")
print(f"t = 200: a = {a_w[199]:.4f}   b = {b_w[199]:.4f}   beta_tilde = {bt_w[199]:.6f}")

assert abs(a_w[0] - 1.0) < 1e-10
assert b_w[99] > 0.95
assert np.all(bt_w <= betas + 1e-15)

## Why a bell curve is the right shape for small steps

Part 8 will model the learned denoiser $p_\theta(x_{t-1} \mid x_t)$ — no $x_0$ input — as a Gaussian, yet Step 3 showed the true $x_0$-free reverse step growing two bumps. The lesson's escape hatch: those bumps came from huge doses. Write the true reverse step as a blend of our exact posteriors, one per possible origin. The component centers disagree through the term $a_t\, x_0$ (here $a_t \cdot 4$ apart, origins $\pm 2$), and $a_t$ shrinks in proportion to $\beta_t$ — while each bell's width $\sqrt{\tilde\beta_t}$ shrinks only like $\sqrt{\beta_t}$, much more slowly. So as the doses get small, the bells crowd together faster than they narrow, and the bumps merge into one.

### Step 8 — Same destruction, tiny doses: watch the bumps merge

We match noise levels. The teaching schedule reaches $\bar\alpha_2 = 0.72$ in two big doses; a constant-dose schedule with $\beta = 0.005$ reaches $\bar\alpha_{66} \approx 0.72$ in 66 tiny ones. For both we condition on $x_t$ near $0.1$ and histogram the part of $x_{t-1}$ the $x_t$ anchor does not explain — the residual $x_{t-1} - b_t x_t = a_t x_0 + \text{noise}$. Big doses: two far-apart bumps. Tiny doses: one merged bell. The spread ratio (mixture spread over single-bell spread $\sqrt{\tilde\beta_t}$) quantifies the difference.

In [ ]:
beta_f = 0.005
T_f = 66
betas_f = np.full(T_f, beta_f)
alphas_f = 1.0 - betas_f
abar_f_full = np.concatenate(([1.0], np.cumprod(alphas_f)))

print(f"abar after 66 tiny doses = {abar_f_full[T_f]:.4f}   (teaching abar_2 = 0.72)")

xf = x0.copy()
for i in range(T_f):
    xf_prev = xf
    eps_f = rng.standard_normal(n)
    xf = np.sqrt(alphas_f[i]) * xf + np.sqrt(betas_f[i]) * eps_f

a_f, b_f, bt_f = posterior_coeffs(T_f, betas_f, alphas_f, abar_f_full)

win_f = np.abs(xf - 0.1) <= 0.4
resid_c = x1[window] - b2 * x2[window]
resid_f = xf_prev[win_f] - b_f * xf[win_f]

ratio_c = resid_c.std() / np.sqrt(bt2)
ratio_f = resid_f.std() / np.sqrt(bt_f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.hist(resid_c, bins=40, density=True, color="#ff7b72")
ax1.set_xlabel("x_1 - b_2 x_2  (part not explained by x_t)")
ax1.set_ylabel("density")
ax1.set_title(f"two big doses: spread {ratio_c:.1f}x one bell")

ax2.hist(resid_f, bins=40, density=True, color="#4ea1ff")
ax2.set_xlabel("x_65 - b_66 x_66  (part not explained by x_t)")
ax2.set_ylabel("density")
ax2.set_title(f"66 tiny doses: spread {ratio_f:.2f}x one bell")

plt.tight_layout()
plt.show()

print(f"spread ratio, big doses:  {ratio_c:.2f}   (far above 1: a two-bump mixture)")
print(f"spread ratio, tiny doses: {ratio_f:.2f}   (close to 1: effectively one bell)")

assert abs(abar_f_full[T_f] - 0.72) < 0.005
assert ratio_c > 3.0
assert ratio_f < 1.3

## The oracle sampler, and the honest gap

Cash in the theorem. If an oracle whispers each sample's clean point $x_0$, generation is three lines: draw $x_T \sim \mathcal{N}(0, \mathbf{I})$, then for $t = T, \dots, 1$ draw $x_{t-1} = \tilde\mu_t + \sqrt{\tilde\beta_t}\, z$ with fresh $z \sim \mathcal{N}(0, \mathbf{I})$, and stop — the formulas themselves set $\tilde\beta_1 = 0$ and $\tilde\mu_1 = x_0$, so the walk lands exactly on the oracle's point. In 2-D each coordinate runs its own copy of the scalar formulas — the "per coordinate" note from the lesson. Every arrow of time in the pictures below is one exact Gaussian draw from today's derivation.

### Step 9 — Build the 8-cluster ring dataset

2,000 clean points: pick one of 8 cluster centers on a circle of radius 4, add a little Gaussian scatter (std 0.15). Each sampler chain will receive one of these points as its oracle's whisper.

In [ ]:
n_pts = 2000
angles = 2 * np.pi * np.arange(8) / 8
centers = 4.0 * np.column_stack([np.cos(angles), np.sin(angles)])

labels = rng.integers(0, 8, size=n_pts)
x0_ring = centers[labels] + 0.15 * rng.standard_normal((n_pts, 2))
radius0 = np.linalg.norm(x0_ring, axis=1)

plt.figure(figsize=(5, 5))
plt.scatter(x0_ring[:, 0], x0_ring[:, 1], s=4, color="#4ea1ff")
plt.xlabel("coordinate 1")
plt.ylabel("coordinate 2")
plt.title("the clean data: 8 clusters on a ring (the oracle's answers)")
plt.axis("equal")
plt.show()

print(f"dataset shape: {x0_ring.shape}   mean radius = {radius0.mean():.3f}")

assert x0_ring.shape == (2000, 2)
assert abs(radius0.mean() - 4.0) < 0.1

### Step 10 — Walk the true posterior from pure noise

Start 2,000 points at pure noise and walk $t = 200, \dots, 1$, each step one Gaussian draw from $q(x_{t-1} \mid x_t, x_0)$ using the oracle's $x_0$. We snapshot six moments along the way. The final assert is the lesson's last practice problem come true: the walk lands on the oracle's point **exactly**, because the $t = 1$ coefficients are $a_1 = 1$, $b_1 = 0$, $\tilde\beta_1 = 0$.

In [ ]:
xt = rng.standard_normal((n_pts, 2))
snapshots = {200: xt.copy()}
snap_at = [150, 100, 50, 25, 0]

for t in range(T, 0, -1):
    a_t, b_t, bt_t = posterior_coeffs(t, betas, alphas, abar_full)
    z = rng.standard_normal((n_pts, 2))
    xt = a_t * x0_ring + b_t * xt + np.sqrt(bt_t) * z
    if t - 1 in snap_at:
        snapshots[t - 1] = xt.copy()

gap = np.max(np.abs(xt - x0_ring))

print(f"largest gap between final samples and the oracle's x0: {gap:.2e}")

assert gap < 1e-9

### Step 11 — Watch the ring reassemble

Read left to right: formless noise grows a hollow center, then a ring, then eight sharp clusters. Reverse-time generation is not a metaphor — it is 200 applications of the formula you derived. The one dishonest ingredient is the oracle: $x_0$ enters each step only through the blend $a_t x_0 + b_t x_t$, so replacing the whisper with a learned guess of $x_0$ (equivalently, of the noise) turns this into a real generative model. That replacement is parts 8-10.

In [ ]:
snap_ts = [200, 150, 100, 50, 25, 0]

fig, axes = plt.subplots(1, 6, figsize=(15, 2.8))
for ax, t in zip(axes, snap_ts):
    pts = snapshots[t]
    ax.scatter(pts[:, 0], pts[:, 1], s=2, color="#4ea1ff")
    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)
    ax.set_title(f"t = {t}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("the oracle sampler: noise reassembling into the ring, one exact Gaussian draw at a time")
plt.tight_layout()
plt.show()

dists = np.linalg.norm(xt[:, None, :] - centers[None, :, :], axis=2)
nearest = dists.min(axis=1)
final_radius = np.linalg.norm(xt, axis=1)

print(f"mean distance from the nearest cluster center = {nearest.mean():.3f}")
print(f"mean radius of final samples = {final_radius.mean():.3f}   (data: 4.0)")

assert nearest.mean() < 0.3
assert abs(final_radius.mean() - 4.0) < 0.2

## Practice

Try each one in the empty cell below it, then reveal the worked solution. These are the same problems as the lesson — redo them here with code as your calculator and checker.

**Problem 1.** The quadratic-coefficient shortcut. Once the exponent reads $-\tfrac12(A x_{t-1}^2 - 2B x_{t-1})$ plus constants, the variance of the resulting Gaussian is $1/A$ — no completing the square needed for the spread alone. Start from $A = \dfrac{\alpha_t}{\beta_t} + \dfrac{1}{1-\bar\alpha_{t-1}}$ and re-derive $\tilde\beta_t$, then confirm numerically at $t = 2$ on the teaching schedule.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Put $A$ over one denominator: $A = \dfrac{\alpha_t(1-\bar\alpha_{t-1}) + \beta_t}{\beta_t(1-\bar\alpha_{t-1})}$ (adding fractions).
- Distribute $\alpha_t$: the numerator becomes $\alpha_t - \alpha_t\bar\alpha_{t-1} + \beta_t$.
- Peel-one-factor identity $\alpha_t\bar\alpha_{t-1} = \bar\alpha_t$: the numerator becomes $\alpha_t - \bar\alpha_t + \beta_t$.
- Use $\alpha_t + \beta_t = 1$: the numerator becomes $1 - \bar\alpha_t$, so $A = \dfrac{1-\bar\alpha_t}{\beta_t(1-\bar\alpha_{t-1})}$.
- Flip the fraction: $\tilde\beta_t = 1/A = \dfrac{\beta_t(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}$ — the full derivation's answer with less work, because the variance never depends on the linear coefficient $B$.

```python
A = alphas3[1] / betas3[1] + 1.0 / (1.0 - abar3_full[1])

print(f"A = {A:.4f}   1/A = {1.0 / A:.6f}   beta_tilde_2 = {bt2:.6f}")
```

**Answer:** $\tilde\beta_t = (1-\bar\alpha_{t-1})\,\beta_t/(1-\bar\alpha_t)$; at $t = 2$, $A = 0.8/0.2 + 1/0.1 = 14$ and $1/A = 0.071429$, matching $\tilde\beta_2$.

</details>

**Problem 2.** A fresh weight table. Take a new two-step schedule: $T = 2$ with $\beta = (0.2,\ 0.5)$. Build $\alpha$ and $\bar\alpha$, then compute the $t = 2$ posterior ingredients $a_2$, $b_2$, and $\tilde\beta_2$, each to four decimals.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Survival factors: $\alpha = (0.8,\ 0.5)$ ($\alpha_t = 1-\beta_t$).
- Running products: $\bar\alpha_1 = 0.8$, $\bar\alpha_2 = 0.8 \times 0.5 = 0.4$.
- Accumulated noise: $1-\bar\alpha_1 = 0.2$ and $1-\bar\alpha_2 = 0.6$.
- Weight on $x_0$: $a_2 = \sqrt{0.8} \times 0.5 / 0.6 = 0.7454$.
- Weight on $x_t$: $b_2 = \sqrt{0.5} \times 0.2 / 0.6 = 0.2357$.
- Spread: $\tilde\beta_2 = 0.2 \times 0.5 / 0.6 = 0.1667$.

```python
betas_p2 = np.array([0.2, 0.5])
alphas_p2 = 1.0 - betas_p2
abar_p2_full = np.concatenate(([1.0], np.cumprod(alphas_p2)))
a_p2, b_p2, bt_p2 = posterior_coeffs(2, betas_p2, alphas_p2, abar_p2_full)

print(f"a_2 = {a_p2:.4f}   b_2 = {b_p2:.4f}   beta_tilde_2 = {bt_p2:.4f}")
```

**Answer:** $a_2 \approx 0.7454$, $b_2 \approx 0.2357$, $\tilde\beta_2 \approx 0.1667$. Step 2's dose was enormous ($\beta_2 = 0.5$), so the noisy point is unreliable and the clean anchor $x_0$ carries most of the weight.

</details>

**Problem 3.** The do-nothing limit. Show that as $\beta_t \to 0$ (with $t > 1$ fixed, so $\bar\alpha_{t-1} < 1$): $a_t \to 0$, $b_t \to 1$, and $\tilde\beta_t \to 0$. Confirm the trend numerically by shrinking a step-2 dose through $0.01,\ 0.001,\ 0.0001$ (with $\beta_1 = 0.1$ held fixed).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $\beta_t \to 0$ forces $\alpha_t \to 1$, hence $\bar\alpha_t \to \bar\alpha_{t-1}$ and $1-\bar\alpha_t \to 1-\bar\alpha_{t-1} > 0$.
- $a_t = \sqrt{\bar\alpha_{t-1}}\,\beta_t/(1-\bar\alpha_t)$: the numerator carries the factor $\beta_t \to 0$ while the denominator stays away from zero, so $a_t \to 0$.
- $b_t \to 1 \cdot (1-\bar\alpha_{t-1})/(1-\bar\alpha_{t-1}) = 1$: the matching numerator and denominator cancel.
- $\tilde\beta_t = (1-\bar\alpha_{t-1})\,\beta_t/(1-\bar\alpha_t) \to 0$: the same vanishing factor $\beta_t$ wins again.

```python
for beta_small in [0.01, 0.001, 0.0001]:
    betas_p3 = np.array([0.1, beta_small])
    alphas_p3 = 1.0 - betas_p3
    abar_p3_full = np.concatenate(([1.0], np.cumprod(alphas_p3)))
    a_p3, b_p3, bt_p3 = posterior_coeffs(2, betas_p3, alphas_p3, abar_p3_full)
    print(f"beta_2 = {beta_small:.4f}:  a = {a_p3:.4f}   b = {b_p3:.4f}   beta_tilde = {bt_p3:.6f}")
```

**Answer:** in the limit the posterior says $x_{t-1} = x_t$ exactly, with zero spread: a step that added no noise is undone by not moving. The formula passes the do-nothing test.

</details>

**Problem 4.** The Markov drop, and its trap. Justify carefully why $q(x_t \mid x_{t-1}, x_0) = q(x_t \mid x_{t-1})$, and then explain why the superficially similar claim "$x_0$ can be dropped from $q(x_{t-1} \mid x_t, x_0)$" is FALSE.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the step-$t$ mechanism as an update rule: $x_t = \sqrt{\alpha_t}\, x_{t-1} + \sqrt{\beta_t}\, \epsilon_t$ with fresh $\epsilon_t \sim \mathcal{N}(0,1)$.
- The rule consumes exactly two inputs — $x_{t-1}$ and the fresh $\epsilon_t$ — and $x_0$ appears nowhere in it. Extra information the mechanism never consults cannot change the outcome's distribution, so revealing $x_0$ on top of $x_{t-1}$ changes nothing: that is the Markov property.
- The trap: $x_{t-1}$ is produced from $x_0$ before $x_t$ exists, so $x_0$ genuinely constrains $x_{t-1}$ even after $x_t$ is seen. The Markov property screens the future off from the deeper past given the present; it never says the past is redundant for guessing the present. The two claims condition in opposite directions.
- The derived weight is the proof that $x_0$ stays informative:

```python
print(f"weight on x0 in the t = 2 posterior: a_2 = {a2:.4f}   (not 0)")
```

**Answer:** dropping $x_0$ is legal only in $q(x_t \mid x_{t-1}, x_0)$, where the conditioned-on $x_{t-1}$ is the sole input of the mechanism that makes $x_t$. In $q(x_{t-1} \mid x_t, x_0)$ the roles reverse — and the weight $a_2 = 0.6776 > 0$ is the proof.

</details>

**Problem 5.** The posterior at $t = 1$, by hand. Using the teaching schedule ($\beta_1 = 0.1$, $\bar\alpha_1 = 0.9$, convention $\bar\alpha_0 = 1$), the clean point $x_0 = 2$, and an observed $x_1 = 1.5$: compute $a_1$, $b_1$, $\tilde\beta_1$, and $\tilde\mu_1$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Convention: $\bar\alpha_0 = 1$, so $1-\bar\alpha_0 = 0$, and $1-\bar\alpha_1 = 0.1$.
- Weight on $x_0$: $a_1 = \sqrt{1} \times 0.1/0.1 = 1$.
- Weight on $x_1$: $b_1 = \sqrt{0.9} \times 0/0.1 = 0$ — the zero factor $1-\bar\alpha_0$ kills the whole term.
- Spread: $\tilde\beta_1 = 0 \times 0.1/0.1 = 0$ — the same zero factor.
- Center: $\tilde\mu_1 = 1 \times 2 + 0 \times 1.5 = 2$.

```python
a_p5, b_p5, bt_p5 = posterior_coeffs(1, betas3, alphas3, abar3_full)
mu_p5 = a_p5 * 2.0 + b_p5 * 1.5

print(f"a_1 = {a_p5:.4f}   b_1 = {b_p5:.4f}   beta_tilde_1 = {bt_p5:.4f}   mu_tilde_1 = {mu_p5:.4f}")
```

**Answer:** $a_1 = 1$, $b_1 = 0$, $\tilde\beta_1 = 0$, $\tilde\mu_1 = 2$: a zero-spread spike at exactly $x_0$. Given the clean point, the question "what was the value before the first dose?" answers itself, and the observed $x_1 = 1.5$ is rightly ignored — the general formula handles its own edge case with no patching.

</details>

**Problem 6.** Why can't we use $q(x_{t-1} \mid x_t)$ directly? Reconstruct the full argument with Bayes' rule, then demonstrate the punchline in code: rerun the two-bump experiment with the data rebalanced to 90% at $+2$ and 10% at $-2$, and watch the bumps change with the data — with no change to the noising recipe.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bayes' rule for the flip: $q(x_{t-1} \mid x_t) = q(x_t \mid x_{t-1})\, q(x_{t-1})/q(x_t)$. The first factor is the forward kernel — fully known, we wrote it ourselves.
- The other two are data averages: $q(x_{t-1}) = \int q(x_{t-1} \mid x_0)\, q(x_0)\, dx_0$ — known closed-form bells weighted by the data distribution $q(x_0)$, and $q(x_t)$ has the same structure.
- The blocker: $q(x_0)$ is the data cloud's rule — the very object the course is trying to learn. The flip is circular: the exact denoiser presupposes the answer.
- The demonstration: rebalance the origins and the reverse conditional changes, with the forward recipe untouched.

```python
x0_re = rng.choice(np.array([-2.0, 2.0]), size=n, p=[0.1, 0.9])
x1_re = np.sqrt(alphas3[0]) * x0_re + np.sqrt(betas3[0]) * rng.standard_normal(n)
x2_re = np.sqrt(alphas3[1]) * x1_re + np.sqrt(betas3[1]) * rng.standard_normal(n)

win_re = np.abs(x2_re - 0.1) <= 0.4
frac_down_re = (x1_re[win_re] < -0.5).mean()

print(f"lower-bump mass, 50/50 data: {frac_down:.3f}")
print(f"lower-bump mass, 10/90 data: {frac_down_re:.3f}   (same recipe, different reverse step)")
```

**Answer:** $q(x_{t-1} \mid x_t)$ requires the marginals $q(x_{t-1})$ and $q(x_t)$, which are data-distribution averages — unknowable without the very thing we want to learn, and generally non-Gaussian (two bumps in the toy). Conditioning on $x_0$ replaces both marginals with the fixed recipe's closed forms, which is exactly why this part's posterior is computable.

</details>

**Problem 7.** Oracle-sampler reasoning. Explain (a) why the oracle sampler's final output always equals the oracle's $x_0$ exactly, and (b) why the run is still deeply informative — what do the intermediate steps demonstrate, and what single replacement turns the oracle sampler into a real generative model?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- (a) At $t = 1$ the coefficients are $a_1 = 1$, $b_1 = 0$, $\tilde\beta_1 = 0$ (Problem 5): the last draw is from a zero-spread Gaussian centered at $x_0$ — a spike. Whatever noisy path the walk took, it lands on the oracle's point. Step 10's assert measured the gap at machine precision.
- (b) Every step $t = T, \dots, 2$ is a draw from the exact posterior $q(x_{t-1} \mid x_t, x_0)$, so the run verifies the whole reverse-time machinery end to end — blend two anchors, add $\sqrt{\tilde\beta_t}$-sized noise — before any neural network exists.
- The single replacement: swap the oracle's $x_0$ for a guess $\hat{x}_0(x_t, t)$ produced by a trained network, and leave every other line of the loop unchanged. $x_0$ enters the step only through the blend $a_t x_0 + b_t x_t$, so any plug-in guess yields a runnable step; parts 8-10 build and train exactly this plug-in (via the equivalent noise-guess $\epsilon_\theta$).

```python
a_last, b_last, bt_last = posterior_coeffs(1, betas, alphas, abar_full)

print(f"t = 1 on the T = 200 schedule: a = {a_last:.6f}   b = {b_last:.6f}   beta_tilde = {bt_last:.6f}")
```

**Answer:** (a) $\tilde\beta_1 = 0$ and $\tilde\mu_1 = x_0$, so the walk ends on the oracle's point exactly. (b) The intermediates prove that noise reassembles into data by plain Gaussian arithmetic, isolating the one missing capability — guessing $x_0$ (equivalently the noise) from $x_t$ — which is precisely what the network of parts 8-10 supplies.

</details>

## Wrap-up

Verified in this notebook: the $x_0$-free reverse conditional $q(x_1 \mid x_2)$ of a two-point dataset really is two-bumped — not Gaussian, and not computable without the data distribution; the exact posterior $q(x_{t-1} \mid x_t, x_0) = \mathcal{N}(\tilde\mu_t,\ \tilde\beta_t)$ matches 200,000 brute-force chains in center, spread, and shape; the hand example gives $\tilde\mu_2 = 1.7066$ and $\tilde\beta_2 = 0.0714$ to four decimals, and the averages-consistency check cancels exactly; across the $T = 200$ schedule the anchor weights hand over from $x_t$ to $x_0$ with $\tilde\beta_t \le \beta_t$ throughout; shrinking the doses merges the two bumps into one bell, legitimizing the Gaussian model for small steps; and the oracle sampler carried pure noise back onto the 8-cluster ring with exact Gaussian draws, landing on the oracle's $x_0$ to machine precision.

Next, Part 8 scores a network that must manage without the oracle: part 5's ELBO, applied to the diffusion chain, decomposes into per-step KL divergences between today's true posterior $q(x_{t-1} \mid x_t, x_0)$ and the learned denoiser $p_\theta(x_{t-1} \mid x_t)$ — one trainable number for the whole chain.